# EDA — Desafio RSNA 2021: Brain Tumor (MGMT Methylation)

**Grupo G3** — TP1 de Tópicos Especiais em Sistemas de Informação

Objetivo deste notebook (Marco 1, Semana 1):
- Ler os dados e metadados do desafio *RSNA-MICCAI Brain Tumor Radiogenomic Classification*
- Conferir a distribuição de classes (MGMT metilado vs. não metilado)
- Contar pacientes, exames e modalidades disponíveis
- Conseguir ler um exame de um paciente do início ao fim, com atenção à orientação de cada série
- Visualizar exemplos por classe
- Salvar evidências (tabelas e figuras) em arquivo, para uso no relatório e no repositório

**Como usar no Kaggle:** crie um Notebook novo na página do desafio, faça upload deste arquivo (File > Import Notebook) ou copie as células, e adicione o dataset da competição em "Add Input". Ao final, use "Save & Run All" para publicar uma versão com todos os outputs salvos.

## 1. Setup e instalação

In [ ]:
# No Kaggle o pydicom geralmente já vem instalado; a linha abaixo garante caso não venha
!pip install -q pydicom

import os
import glob
import random
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Caminho padrão do dataset quando adicionado via "Add Input" no Kaggle
DATA_DIR = "/kaggle/input/competitions/rsna-miccai-brain-tumor-radiogenomic-classification"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
LABELS_CSV = os.path.join(DATA_DIR, "train_labels.csv")

MODALITIES = ["FLAIR", "T1w", "T1wCE", "T2w"]

# Pasta de saída — tudo que este notebook gera fica organizado aqui
OUT_DIR = "outputs_eda_g3"
os.makedirs(OUT_DIR, exist_ok=True)

print("DATA_DIR existe?", os.path.exists(DATA_DIR))

## 2. Rótulos e distribuição de classes

O arquivo `train_labels.csv` traz o ID do paciente (`BraTS21ID`) e o rótulo (`MGMT_value`: 0 = não metilado, 1 = metilado).

Isso já responde uma pergunta central do relatório: **o quão desbalanceado é o problema?**

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
print(f"Total de pacientes no train_labels.csv: {len(labels_df)}")
labels_df.head()

In [ ]:
class_counts = labels_df["MGMT_value"].value_counts().sort_index()
class_pct = labels_df["MGMT_value"].value_counts(normalize=True).sort_index() * 100

print("Contagem por classe:")
print(class_counts)
print("\nPercentual por classe:")
print(class_pct.round(2))

# Salvar tabela de distribuição — número que vai direto pra Metodologia do artigo
class_summary = pd.DataFrame({
    "MGMT_value": class_counts.index,
    "n_pacientes": class_counts.values,
    "percentual": class_pct.round(2).values
})
class_summary.to_csv(os.path.join(OUT_DIR, "distribuicao_classes.csv"), index=False)

plt.figure(figsize=(5, 4))
class_counts.plot(kind="bar", color=["steelblue", "indianred"])
plt.xticks([0, 1], ["MGMT não metilado (0)", "MGMT metilado (1)"], rotation=20)
plt.ylabel("Número de pacientes")
plt.title("Distribuição de classes — MGMT")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "distribuicao_classes.png"), dpi=150)
plt.show()

# ANOTAR NO RELATÓRIO: essa distribuição costuma sair perto de 47%/53% neste desafio —
# ou seja, aproximadamente balanceada. Diferente de outros desafios RSNA (ex: mamografia,
# ~2% positivo), aqui a dificuldade dominante NÃO é desbalanceamento, e sim o sinal
# genético fraco (ver observações do TP1, §5, sobre este desafio especificamente).

## 3. Contagem de pacientes, exames e modalidades

Cada paciente tem uma pasta com 4 subpastas (uma por modalidade de RM), cada uma contendo os cortes DICOM daquela sequência.

In [ ]:
patient_ids = sorted(os.listdir(TRAIN_DIR))
print(f"Pastas de paciente encontradas em train/: {len(patient_ids)}")
print("Exemplos de IDs:", patient_ids[:5])

In [ ]:
# Para cada paciente, checar quais modalidades existem e quantos cortes (slices) tem cada uma
rows = []
for pid in patient_ids:
    patient_dir = os.path.join(TRAIN_DIR, pid)
    row = {"patient_id": pid}
    for mod in MODALITIES:
        mod_dir = os.path.join(patient_dir, mod)
        n_slices = len(glob.glob(os.path.join(mod_dir, "*.dcm"))) if os.path.isdir(mod_dir) else 0
        row[f"n_slices_{mod}"] = n_slices
    rows.append(row)

slices_df = pd.DataFrame(rows)

# Salvar — evidência de completude dos dados, útil pra decisão de agregação (§4.2 do TP1)
slices_df.to_csv(os.path.join(OUT_DIR, "contagem_cortes_por_paciente.csv"), index=False)

slices_df.head()

In [ ]:
# Quantos pacientes têm as 4 modalidades completas (nenhuma pasta vazia)?
has_all = (slices_df[[f"n_slices_{m}" for m in MODALITIES]] > 0).all(axis=1)
print(f"Pacientes com as 4 modalidades presentes: {has_all.sum()} de {len(slices_df)}")

# Estatística do número de cortes por modalidade (ajuda a decidir estratégia de agregação — ver §4.2 do TP1)
stats_df = slices_df[[f"n_slices_{m}" for m in MODALITIES]].describe()
stats_df.to_csv(os.path.join(OUT_DIR, "estatisticas_cortes.csv"))
stats_df

## 4. Ler um exame do início ao fim (metadados DICOM + orientação da série)

Aqui atendemos o requisito de "leitura de DICOM e extração de metadados relevantes" do TP1: modalidade, espaçamento de pixel, e demais campos disponíveis.

**Ajuste importante:** cada série DICOM carrega a tag `ImageOrientationPatient` (IOP) — dois vetores que descrevem a orientação da linha e da coluna da imagem no espaço do paciente. A partir dela dá pra calcular o vetor normal ao plano (produto vetorial) e classificar a série como aproximadamente **axial**, **sagital** ou **coronal**. Isso é necessário porque, neste dataset, as 4 modalidades de um mesmo paciente podem ter sido adquiridas em planos diferentes — pegar "o corte do meio" pela posição no arquivo não garante que estamos olhando a mesma região anatômica em todas elas.

In [ ]:
def classify_plane(iop):
    """Classifica o plano de aquisição (axial/sagital/coronal) a partir do ImageOrientationPatient.
    iop: lista/array com 6 valores (3 do vetor linha + 3 do vetor coluna).
    Retorna a string do plano com maior componente no vetor normal (produto vetorial linha x coluna).
    """
    if iop is None or len(iop) != 6:
        return "desconhecido"
    row_vec = np.array(iop[0:3], dtype=float)
    col_vec = np.array(iop[3:6], dtype=float)
    normal = np.cross(row_vec, col_vec)
    axis = np.argmax(np.abs(normal))
    # eixo 0 (x, esquerda-direita) dominante -> corte sagital
    # eixo 1 (y, anterior-posterior) dominante -> corte coronal
    # eixo 2 (z, cabeça-pés) dominante -> corte axial
    return {0: "sagital", 1: "coronal", 2: "axial"}[axis]


def read_series(patient_id, modality, data_dir=TRAIN_DIR):
    """Lê todos os arquivos DICOM de uma modalidade de um paciente.
    Ordena por posição física (ImagePositionPatient projetada no vetor normal) quando disponível;
    cai para ordenação pelo número no nome do arquivo caso a tag não exista.
    """
    series_dir = os.path.join(data_dir, patient_id, modality)
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    slices = [pydicom.dcmread(f) for f in files]

    iop = getattr(slices[0], "ImageOrientationPatient", None) if slices else None
    if iop is not None:
        row_vec = np.array(iop[0:3], dtype=float)
        col_vec = np.array(iop[3:6], dtype=float)
        normal = np.cross(row_vec, col_vec)
        try:
            slices.sort(key=lambda ds: float(np.dot(np.array(ds.ImagePositionPatient, dtype=float), normal)))
        except Exception:
            slices.sort(key=lambda ds: int(''.join(filter(str.isdigit, os.path.basename(ds.filename)))))
    else:
        slices.sort(key=lambda ds: int(''.join(filter(str.isdigit, os.path.basename(ds.filename)))))

    return slices


sample_patient = patient_ids[0]
sample_series = read_series(sample_patient, "FLAIR")
ds = sample_series[len(sample_series) // 2]  # corte do meio, agora em ordem física correta
iop_sample = getattr(ds, "ImageOrientationPatient", None)

metadata_info = {
    "paciente": sample_patient,
    "modalidade": "FLAIR",
    "n_cortes": len(sample_series),
    "dimensoes": str(ds.pixel_array.shape),
    "pixel_spacing": str(getattr(ds, "PixelSpacing", "não informado")),
    "slice_thickness": str(getattr(ds, "SliceThickness", "não informado")),
    "modality_tag": str(getattr(ds, "Modality", "não informado")),
    "rows": ds.Rows,
    "columns": ds.Columns,
    "image_orientation_patient": str(iop_sample),
    "plano_estimado": classify_plane(iop_sample),
}

print("Paciente:", sample_patient, "| Modalidade: FLAIR | Cortes:", len(sample_series))
for k, v in metadata_info.items():
    print(f"{k}: {v}")

# Salvar — evidência de que a leitura DICOM (incluindo orientação) foi feita e verificada
pd.DataFrame([metadata_info]).to_csv(os.path.join(OUT_DIR, "metadados_exemplo.csv"), index=False)

In [ ]:
# Checar o plano de cada modalidade para o mesmo paciente — é aqui que aparece se
# FLAIR/T1w/T1wCE/T2w foram de fato adquiridas em planos diferentes.
orientation_rows = []
for mod in MODALITIES:
    series = read_series(sample_patient, mod)
    iop = getattr(series[0], "ImageOrientationPatient", None) if series else None
    orientation_rows.append({
        "paciente": sample_patient,
        "modalidade": mod,
        "plano_estimado": classify_plane(iop),
        "n_cortes": len(series),
    })

orientation_df = pd.DataFrame(orientation_rows)
orientation_df.to_csv(os.path.join(OUT_DIR, "orientacao_por_modalidade_exemplo.csv"), index=False)
orientation_df

## 5. Exemplos visuais por classe (agora com o plano identificado em cada título)

Requisito do Marco 1: mostrar exemplos visuais por classe. Aqui pegamos um paciente MGMT=0 e um MGMT=1 e exibimos o corte central de cada modalidade, com o plano de aquisição indicado no título — em vez de assumir que todas as modalidades estão no mesmo plano.

In [ ]:
def get_middle_slice_info(patient_id, modality):
    series = read_series(patient_id, modality)
    ds = series[len(series) // 2]
    iop = getattr(series[0], "ImageOrientationPatient", None) if series else None
    return ds.pixel_array, classify_plane(iop)

# Pegar um paciente de cada classe que também tenha ID presente nas pastas de imagem
ids_com_imagem = set(patient_ids)
# BraTS21ID no CSV costuma vir como inteiro; as pastas usam string com zero à esquerda (ex: "00000")
labels_df["patient_id_str"] = labels_df["BraTS21ID"].astype(str).str.zfill(5)
labels_df_valid = labels_df[labels_df["patient_id_str"].isin(ids_com_imagem)]

patient_0 = labels_df_valid[labels_df_valid["MGMT_value"] == 0]["patient_id_str"].iloc[0]
patient_1 = labels_df_valid[labels_df_valid["MGMT_value"] == 1]["patient_id_str"].iloc[0]

fig, axes = plt.subplots(2, len(MODALITIES), figsize=(4 * len(MODALITIES), 8))
for row, (pid, label) in enumerate([(patient_0, "MGMT = 0"), (patient_1, "MGMT = 1")]):
    for col, mod in enumerate(MODALITIES):
        try:
            img, plane = get_middle_slice_info(pid, mod)
            axes[row, col].imshow(img, cmap="gray")
            axes[row, col].set_title(f"{mod} ({plane}) — {label}")
        except Exception as e:
            axes[row, col].text(0.5, 0.5, "erro ao ler", ha="center")
            axes[row, col].set_title(f"{mod} — {label}")
        axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "exemplos_visuais_por_classe.png"), dpi=150)
plt.show()

## 6. Amostragem estratificada (se necessário)

O dataset completo é grande. O TP1 permite (e espera) trabalhar com uma amostra estratificada, desde que documentada, com semente fixa e contagem por classe.

In [ ]:
N_POR_CLASSE = 30  # ajustar conforme viabilidade de tempo/armazenamento do grupo

amostra_0 = labels_df_valid[labels_df_valid["MGMT_value"] == 0].sample(
    n=min(N_POR_CLASSE, (labels_df_valid["MGMT_value"] == 0).sum()), random_state=SEED
)
amostra_1 = labels_df_valid[labels_df_valid["MGMT_value"] == 1].sample(
    n=min(N_POR_CLASSE, (labels_df_valid["MGMT_value"] == 1).sum()), random_state=SEED
)
amostra_df = pd.concat([amostra_0, amostra_1]).reset_index(drop=True)

print("Critério: amostragem estratificada por classe, semente fixa =", SEED)
print(amostra_df["MGMT_value"].value_counts())

amostra_df.to_csv(os.path.join(OUT_DIR, "amostra_g3.csv"), index=False)
print(f"Amostra salva em {OUT_DIR}/amostra_g3.csv (documentar isso na Metodologia do artigo)")

## 7. Conferência final dos arquivos salvos

In [ ]:
print(f"Arquivos gerados em '{OUT_DIR}/':")
for f in sorted(os.listdir(OUT_DIR)):
    print(" -", f)

# No Kaggle, use "Save & Run All (Commit)" para publicar uma versão do notebook
# com esta pasta de outputs anexada e baixável — é essa versão que deve ir pro repositório.